In [0]:
%python
import sys
sys.path.append('/Workspace/Users/motloungmiya@gmail.com/medallion-ecommerce-scd')

from config.pipeline_config import *

print("✓ Configuration loaded")

In [0]:
%python

# ── Create Unity Catalog Volume and directories ──────────────
# STUDY NOTE: Unity Catalog Volumes provide governed file storage.
#   First create the Volume, then create subdirectories within it.

# Create the Volume if it doesn't exist
spark.sql("""
          CREATE VOLUME IF NOT EXISTS workspace.default.olist_data
          """)

print("✓ Volume workspace.default.olist_data ready")


# Create layer subdirectories
dirs_to_create = [BRONZE_PATH, SILVER_PATH, GOLD_PATH]

for path in dirs_to_create:
    dbutils.fs.mkdirs(path)
    print(f"✓ Created: {path}")


In [0]:
CREATE TABLE IF NOT EXISTS default.pipeline_watermarks (
    table_name      STRING,
    last_extracted_at TIMESTAMP,
    batch_id        STRING,
    rows_extracted  LONG,
    status          STRING,
    updated_at      TIMESTAMP
) USING DELTA;

-- Query the table and view its columns
SELECT * FROM default.pipeline_watermarks;

In [0]:
%python
# ── Verify all paths exist ───────────────────────────────────
# STUDY NOTE: dbutils.fs.ls() lists directory contents.
#   We use it here as a health check — if a path doesn't
#   exist this will raise an error, which is what we want.
#   Fail fast → easier to debug.

print("Verifying directory structure...\n")

base_listing = dbutils.fs.ls(f"{BASE_PATH}/")

for item in base_listing:
    print(f"  {item.path}")
    sub_listing = dbutils.fs.ls(item.path)
    for sub in sub_listing:
        print(f"    └── {sub.path}")

In [0]:
%python

# ── Print config summary ─────────────────────────────────────
# Confirms what will be used by downstream notebooks

print("=" * 50)
print("Pipeline Configuration Summary")
print("=" * 50)
print(f"Bronze path : {BRONZE_PATH}")
print(f"Silver path : {SILVER_SCD_TABLE}")
print(f"watermark   : {WATERMARK_TABLE}")
print(f"Tracked  : {SCD_TRACKED_COLUMNS}")
print("=" * 50)
print("Setup complete. Ready for 01_bronze_ingest.")